<a href="https://colab.research.google.com/github/MiguelCortezPino/etl-data-pipeline-1701472022/blob/main/notebooks/e_bodegas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd


In [ ]:
url = "https://raw.githubusercontent.com/MiguelCortezPino/etl-data-pipeline-1701472022/refs/heads/main/data/raw/E_bodegas.csv"

df = pd.read_csv(url)

df.head()

,id_bodega,bodega,ubicacion,capacidad_m2
0,BOD100,Central 0,Usulután,1292 m2
1,BOD101,Sur 1,San Miguel,2047
2,BOD102,Central 2,Sonsonate,651 m2
3,BOD103,Occidente 3,San Miguel,2250
4,BOD104,Sur 4,Santa Ana,239


In [ ]:
#Exploracion de datos
df.shape
df.columns
df.info()
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   id_bodega     18 non-null     object
 1   bodega        20 non-null     object
 2   ubicacion     20 non-null     object
 3   capacidad_m2  20 non-null     object
dtypes: object(4)
memory usage: 772.0+ bytes


,0
id_bodega,2
bodega,0
ubicacion,0
capacidad_m2,0


In [ ]:
#limpieza de datos

# Reemplazar 'error' con NaN en ambas columnas
df['id_bodega'] = df['id_bodega'].replace('error', pd.NA)
df['ubicacion'] = df['ubicacion'].replace('error', pd.NA)

# Verificar la cantidad de valores nulos después de la sustitución
print("Valores nulos después de reemplazar 'error':")
print(df.isnull().sum())

# Mostrar los valores únicos de cada columna para identificar patrones
print("\nValores únicos en la columna 'id_bodega':")
print(df['id_bodega'].unique())

print("\nValores únicos en la columna 'ubicacion':")
print(df['ubicacion'].unique())

Valores nulos después de reemplazar 'error':
id_bodega       2
bodega          0
ubicacion       0
capacidad_m2    0
dtype: int64

Valores únicos en la columna 'id_bodega':
['BOD100' 'BOD101' 'BOD102' 'BOD103' 'BOD104' 'BOD105' 'BOD106' 'BOD107'
 'BOD108' 'BOD109' nan 'BOD111' 'BOD112' 'BOD113' 'BOD115' 'BOD116'
 'BOD117']

Valores únicos en la columna 'ubicacion':
['Usulután' 'San Miguel' 'Sonsonate' 'Santa Ana' 'La Libertad'
 'San Salvador']


In [ ]:
#remplazar numero malos

# Reemplazar 'text_52' y 'text_74' con NA en las columnas correspondientes
df['id_bodega'] = df['id_bodega'].replace('text_52', pd.NA)
df['ubicacion'] = df['ubicacion'].replace('text_74', pd.NA)

# At this point, the columns should not be converted to numeric as they are string/object types.
# Removing the pd.to_numeric calls for id_bodega and ubicacion.

# Verificar los valores nulos después de la limpieza
print("Valores nulos después de limpiar 'text_52' y 'text_74' y convertir a numérico:")
print(df.isnull().sum())

# Mostrar los valores únicos de cada columna para verificar
print("\nValores únicos en la columna 'id_bodega' después de la limpieza:")
print(df['id_bodega'].unique())

print("\nValores únicos en la columna 'ubicacion' después de la limpieza:")
print(df['ubicacion'].unique())

Valores nulos después de limpiar 'text_52' y 'text_74' y convertir a numérico:
id_bodega       20
bodega           0
ubicacion       20
capacidad_m2     0
dtype: int64

Valores únicos en la columna 'id_bodega' después de la limpieza:
[nan]

Valores únicos en la columna 'ubicacion' después de la limpieza:
[nan]


In [ ]:
# Separar datos validos y rechazados

# Los datos válidos serán aquellos donde 'id_bodega' y 'ubicacion' no son nulos
df_validos = df.dropna(subset=['id_bodega', 'ubicacion'])


print("Datos Válidos (primeras 5 filas):")
display(df_validos.head())
print(f"Forma de df_validos: {df_validos.shape}\n")

Datos Válidos (primeras 5 filas):


,id_bodega,bodega,ubicacion,capacidad_m2


Forma de df_validos: (0, 4)



In [ ]:
# Los datos rechazados serán aquellos donde 'id_bodega' o 'ubicacion' son nulos
df_rechazados = df[df['id_bodega'].isnull() | df['ubicacion'].isnull()].copy()

print("Datos Rechazados (primeras 5 filas):")
display(df_rechazados.head())
print(f"Forma de df_rechazados: {df_rechazados.shape}")

Datos Rechazados (primeras 5 filas):


,id_bodega,bodega,ubicacion,capacidad_m2
10,NaN,Sur 10,San Salvador,299
14,NaN,Norte 14,San Salvador,1635


Forma de df_rechazados: (2, 4)


In [ ]:
#motivo de rechazo

# Función para determinar el motivo de rechazo
def obtener_motivo_rechazo(row):
    motivos = []
    if pd.isna(row['id']):
        motivos.append('id_nulo')
    if pd.isna(row['ubicacion']):
        motivos.append('ubicacion_nula')
    return ', '.join(motivos) if motivos else 'No_rechazado' # Aunque en df_rechazados siempre habrá motivo

# Aplicar la función para crear la columna 'motivo_rechazo'
df_rechazados['motivo_rechazo'] = df_rechazados.apply(obtener_motivo_rechazo, axis=1)

print("Datos Rechazados con motivo (primeras 5 filas):")
display(df_rechazados.head(5))

print("Conteo de motivos de rechazo:")
print(df_rechazados['motivo_rechazo'].value_counts())

Datos Rechazados con motivo (primeras 5 filas):


,id,ubicacion,motivo_rechazo
0,NaN,16.0,id_nulo
1,NaN,NaN,"id_nulo, ubicacion_nula"
2,NaN,16.0,id_nulo
3,NaN,16.0,id_nulo
4,NaN,NaN,"id_nulo, ubicacion_nula"


Conteo de motivos de rechazo:
motivo_rechazo
id_nulo, ubicacion_nula    1943
ubicacion_nula              490
id_nulo                     455
Name: count, dtype: int64


In [ ]:
df_validos.to_csv("E_bodegas_cureted.csv", index=False)

df_rechazados.to_csv("E_bodega_rejects.csv", index=False)

print("Archivos guardados con éxito: 'E_bodegas_cureted.csv' y 'E_bodega_rejects.csv'")

Archivos guardados con éxito: 'E_bodegas_cureted.csv' y 'E_bodega_rejects.csv'
